In [58]:
import numpy as np
import h5py
import cv2
from joblib import load
import matplotlib.pyplot as plt
%matplotlib widget
import time
%load_ext autoreload
import os
%autoreload 2

from ipywidgets import Video, Image, VBox, Text
from sidecar import Sidecar
from IPython.display import display
from scipy.interpolate import interp1d
from scipy import interpolate
import numpy as np
import pandas as pd
import os
import cv2
import base64
import warnings

from ipywidgets import interact, interactive, fixed, interact_manual, FloatSlider, Layout, AppLayout
import ipywidgets as widgets


# Our custom-made imports
import sys
sys.path.append('./lib/')
from DataManagement import SplitdataManager, Filepaths, Calibration

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Set the filepaths

In [1]:
main_folder = '/home/liam/Data/consistency_test/'

In [9]:
my_path = os.path.join(main_folder, 'GUI_save_folder_' + 'liam', 'test.h5')
tatsuo_path = os.path.join(main_folder, 'GUI_save_folder_' + 'tatsuo', 'test.h5')
manisha_path = os.path.join(main_folder, 'GUI_save_folder_' + 'manisha', 'test.h5')
thomas_path = os.path.join(main_folder, 'GUI_save_folder_' + 'thomas', 'test.h5')
rafael_path = os.path.join(main_folder, 'GUI_save_folder_' + 'Rafael', 'test.h5')

paths = [my_path, tatsuo_path, manisha_path, thomas_path, rafael_path]
labellers = ['liam', 'tatsuo', 'manisha', 'thomas', 'rafael']

In [10]:
# load the results

annotated_frames_list = []
missing_bps_list = []
tracks_3D_list = []
tracks_imCoords_list = []

for i, path in enumerate(paths):
    
    with h5py.File(path, 'r') as hf:
        annotated_frames = hf['annotated_frames'][:]
        missing_bps = hf['missing_bps'][:]
        tracks_3D = hf['tracks_3D'][:]
        tracks_imCoords = hf['tracks_imCoords'][:]
        
    annotated_frames_list.append(annotated_frames)
    missing_bps_list.append(missing_bps)
    tracks_3D_list.append(tracks_3D)
    tracks_imCoords_list.append(tracks_imCoords)

In [11]:
# define the frame nubmers of interest

fIdxs = [0, 50000, 100000, 250000, 350000, 400000, 450000, 500000]

In [ ]:
# Each frame, each cam, each fish, each bp has been measured by 5 people
#
# This means each image coordinate has the following information,
# (1) frameIdx
# (2) fishIdx
# (3) bpIdx
# (4) camIdx
# (5) labellerIdx
# (6) X coord
# (7) Y coord

In [65]:
# Make a dataFrame to hold the results
consistency_df = pd.DataFrame(columns=['frameIdx', 'fishIdx', 'bpIdx', 'camIdx', 'labellerIdx', 'x', 'y'])

for f in fIdxs:

    for camIdx in range(3):
        for fishIdx in range(2):
            for bpIdx in range(3):
                label_coords = []
                for personIdx in range(5):
                    coords = tracks_imCoords_list[personIdx][camIdx, f, fishIdx, bpIdx]
                    x = coords[0]
                    y = coords[1]
                    dic = {'frameIdx':f, 'fishIdx':fishIdx, 'bpIdx':bpIdx,
                           'camIdx':camIdx, 'labellerIdx':personIdx, 'x':x, 'y':y}
                    consistency_df = consistency_df.append(dic, ignore_index=True)

In [66]:
consistency_df

,frameIdx,fishIdx,bpIdx,camIdx,labellerIdx,x,y
0,0.0,0.0,0.0,0.0,0.0,965.690000,847.540000
1,0.0,0.0,0.0,0.0,1.0,964.020000,848.020000
2,0.0,0.0,0.0,0.0,2.0,965.690000,847.540000
3,0.0,0.0,0.0,0.0,3.0,965.690000,847.540000
4,0.0,0.0,0.0,0.0,4.0,965.690000,847.540000
...,...,...,...,...,...,...,...
715,500000.0,1.0,2.0,2.0,0.0,684.020569,855.924377
716,500000.0,1.0,2.0,2.0,1.0,687.790000,855.820000
717,500000.0,1.0,2.0,2.0,2.0,684.020000,855.920000
718,500000.0,1.0,2.0,2.0,3.0,684.020000,855.920000


In [87]:
# Make a dataFrame to hold the results
consistency_df = pd.DataFrame(columns=['frameIdx', 'fishIdx', 'bpIdx', 'camIdx', 'x_mean', 'y_mean', 'x_stdev', 'y_stdev'])

for f in fIdxs:

    for camIdx in range(3):
        for fishIdx in range(2):
            for bpIdx in range(3):
                
                # for this point, examine what everyone did
                label_coords = []
                peoples_x_estimates = []
                peoples_y_estimates = []
                for personIdx in range(5):
                    coords = tracks_imCoords_list[personIdx][camIdx, f, fishIdx, bpIdx]
                    x = coords[0]
                    y = coords[1]
                    peoples_x_estimates.append(x)
                    peoples_y_estimates.append(y)
                # find means and stddev
                x_mean = np.mean(peoples_x_estimates)
                y_mean = np.mean(peoples_y_estimates)
                x_stdev = np.std(peoples_x_estimates)
                y_stdev = np.std(peoples_y_estimates)
                dic = {'frameIdx':f, 'fishIdx':fishIdx, 'bpIdx':bpIdx, 'camIdx':camIdx,
                       'x_mean':x_mean, 'y_mean':y_mean, 'x_stdev':x_stdev, 'y_stdev':y_stdev}
                consistency_df = consistency_df.append(dic, ignore_index=True)
                


In [93]:

pd.options.display.max_rows = None
display(consistency_df)

,frameIdx,fishIdx,bpIdx,camIdx,x_mean,y_mean,x_stdev,y_stdev
0,0.0,0.0,0.0,0.0,965.356000,847.636000,0.668000,0.192000
1,0.0,0.0,1.0,0.0,962.276794,847.264954,0.673398,0.314022
2,0.0,0.0,2.0,0.0,961.938134,846.129156,0.177895,0.809844
3,0.0,1.0,0.0,0.0,956.425698,776.744000,150.239145,147.982253
4,0.0,1.0,1.0,0.0,886.447316,853.363044,1.142481,0.263425
5,0.0,1.0,2.0,0.0,902.000844,857.951872,0.387776,0.589343
6,0.0,0.0,0.0,1.0,351.710000,630.168000,0.527636,1.168493
7,0.0,0.0,1.0,1.0,355.034000,613.552000,0.473269,1.024000
8,0.0,0.0,2.0,1.0,358.928000,581.776000,0.004000,1.162129
9,0.0,1.0,0.0,1.0,416.218000,696.178000,1.117540,0.990200


In [102]:
# examining individual rows that look bad

fIdx = 100000
fishIdx = 0
bpIdx = 1 
camIdx = 0

vals = []
for personIdx in range(5):
    vals.append(tracks_imCoords_list[personIdx][camIdx, fIdx, fishIdx, bpIdx])


In [103]:
vals

[array([717.65819746, 816.34137062]),
 array([717.66, 816.34]),
 array([705.86, 819.62]),
 array([719.05472432, 816.40820817]),
 array([717.66, 816.34])]

In [104]:
labellers

['liam', 'tatsuo', 'manisha', 'thomas', 'rafael']